# Day 02 — ML performance: ROC AUC, confusion matrix, so sánh ROI

**Slide tham khảo chi tiết:** [day02_reference_pack.zip](_static/slides/day02_reference_pack.zip)

## Mục tiêu bài học

- tạo được feature sets theo nhóm clinical, intra, ring1, ring3, ring5
- chạy train test split với pipeline cơ bản
- tính ROC AUC, confusion matrix, sensitivity, specificity
- so sánh các ROI bằng cùng một quy trình đánh giá

## Nội dung

Buổi này tương ứng phần trung tâm của báo cáo: mô hình dự đoán và bảng hiệu năng ban đầu. Học sinh cần hiểu logic đánh giá đúng cách hơn là thuộc cú pháp từng hàm.

## Sản phẩm sau bài học

- `performance_summary.csv`
- ít nhất 1 hình ROC
- 1 confusion matrix
- 1 đoạn nhận xét ROI nào tốt nhất

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
plt.rcParams["figure.figsize"] = (6,4)

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO_NAME = "YOUR_REPO_NAME"
BRANCH = "main"


def download_from_github(rel_path: str) -> Path:
    import urllib.request
    url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{rel_path}"
    out = Path(rel_path).name
    urllib.request.urlretrieve(url, out)
    return Path(out)


def load_csv(rel_path: str) -> pd.DataFrame:
    p = Path(rel_path)
    if p.exists():
        return pd.read_csv(p)
    try:
        p2 = download_from_github(rel_path)
        return pd.read_csv(p2)
    except Exception as e:
        raise FileNotFoundError(f"Không tìm thấy {rel_path}. Hãy chỉnh đúng GITHUB_USER/REPO_NAME hoặc upload file thủ công.") from e


df = load_csv("data/nsclc_egfr_radiomics_simplified.csv")
df.head()

## 1. Tạo nhóm cột đặc trưng

Các cột được gom theo đúng logic báo cáo: clinical, intra, ring1, ring3, ring5.

In [ ]:

clinical_cols = ["age", "sex", "smoking_status", "histology", "stage", "tumor_size_mm", "tumor_volume_cm3", "tp53_mutation"]
intra_cols = [c for c in df.columns if c.startswith("intra_")]
ring1_cols = [c for c in df.columns if c.startswith("ring1_")]
ring3_cols = [c for c in df.columns if c.startswith("ring3_")]
ring5_cols = [c for c in df.columns if c.startswith("ring5_")]

feature_sets = {
    "clinical": clinical_cols,
    "intra": intra_cols,
    "ring1": ring1_cols,
    "ring3": ring3_cols,
    "ring5": ring5_cols,
}

{k: len(v) for k, v in feature_sets.items()}


## 2. Hàm tạo pipeline và đánh giá

Điểm quan trọng là đặt các bước tiền xử lý vào trong pipeline để tránh leakage.

In [ ]:

def split_numeric_categorical(cols):
    num = [c for c in cols if pd.api.types.is_numeric_dtype(df[c]) and df[c].dtype != object]
    cat = [c for c in cols if c not in num]
    return num, cat


def make_pipeline(cols):
    num_cols, cat_cols = split_numeric_categorical(cols)
    pre = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ])
    model = LogisticRegression(max_iter=2000)
    return Pipeline([
        ("pre", pre),
        ("model", model)
    ])


def evaluate_feature_set(cols, random_state=42):
    X = df[cols]
    y = df["egfr_mutation"].astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=random_state
    )
    pipe = make_pipeline(cols)
    pipe.fit(X_train, y_train)
    pred_prob = pipe.predict_proba(X_test)[:, 1]
    pred_label = (pred_prob >= 0.5).astype(int)
    auc = roc_auc_score(y_test, pred_prob)
    cm = confusion_matrix(y_test, pred_label)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    return {
        "pipe": pipe,
        "y_test": y_test,
        "pred_prob": pred_prob,
        "pred_label": pred_label,
        "auc": auc,
        "cm": cm,
        "sensitivity": sensitivity,
        "specificity": specificity,
    }


## 3. Chạy từng feature set và lập bảng so sánh

In [ ]:

results = {}
rows = []
for name, cols in feature_sets.items():
    out = evaluate_feature_set(cols)
    results[name] = out
    rows.append([
        name,
        len(cols),
        round(out["auc"], 3),
        round(out["sensitivity"], 3),
        round(out["specificity"], 3)
    ])

performance_summary = pd.DataFrame(rows, columns=["feature_set", "n_features", "auc", "sensitivity", "specificity"]).sort_values("auc", ascending=False)
performance_summary


## 4. Vẽ ROC cho các ROI

Mục tiêu của hình này là nhìn nhanh ROI nào đang tách nhóm tốt hơn.

In [ ]:

out_dir = Path("outputs/day02")
out_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots()
for name in ["clinical", "intra", "ring1", "ring3", "ring5"]:
    y_test = results[name]["y_test"]
    pred_prob = results[name]["pred_prob"]
    fpr, tpr, _ = roc_curve(y_test, pred_prob)
    ax.plot(fpr, tpr, label=f"{name} (AUC={results[name]['auc']:.3f})")
ax.plot([0,1], [0,1], linestyle="--", color="gray")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC comparison across feature sets")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(out_dir / "roc_comparison.png", dpi=150)
plt.show()


## 5. Confusion matrix cho mô hình tốt nhất

In [ ]:

best_name = performance_summary.iloc[0]["feature_set"]
fig, ax = plt.subplots()
ConfusionMatrixDisplay(results[best_name]["cm"]).plot(ax=ax, colorbar=False)
ax.set_title(f"Confusion matrix — {best_name}")
fig.tight_layout()
fig.savefig(out_dir / f"cm_{best_name}.png", dpi=150)
plt.show()


## 6. Đọc kết quả

Ở bước này chưa cần kết luận chắc chắn mô hình nào tốt nhất cho nghiên cứu. Chỉ cần trả lời được: trong cùng một quy trình đánh giá, ROI nào đang cho AUC tốt hơn.

In [ ]:

performance_summary.to_csv(out_dir / "performance_summary.csv", index=False)
performance_summary


## 7. Mẫu diễn giải ngắn

- Trong lần train test split minh họa, các feature sets ROI cho AUC khác nhau.
- Nếu ring1 hoặc intra đứng đầu, có thể xem đây là ứng viên để mang sang bước kiểm tra độ ổn định ở Day 03.
- Kết quả của một lần split chưa đủ, vì vậy cần cross validation và bootstrap trước khi kết luận.

## 8. Tự kiểm tra

1. Vì sao không dùng cùng một bộ dữ liệu để vừa train vừa đánh giá?  
2. Trong pipeline trên, bước nào giúp tránh leakage?  
3. AUC và confusion matrix khác nhau ở điểm nào?  
4. Vì sao nên dùng cùng một random_state khi so sánh nhiều feature set?

## Nối sang buổi sau

Day 03 sẽ trả lời câu hỏi quan trọng hơn: kết quả của Day 02 có ổn định hay chỉ là may mắn do một lần chia dữ liệu.